# DPA — Volume (Part 1): Network Traffic Analytics with Spark SQL

**Adapted from your uploaded `spark-sql-tp.ipynb` (class TP).** We use the same techniques—DataFrame creation and schemas, `select`, `withColumn`, filters, `distinct`, `groupBy`, `agg`, date functions, temporary SQL views, joins, Parquet writes and `explain`—on the cybersecurity project datasets.

**Working sub-question (editable):** *How do normal, attempted, and malicious network flows differ by attack label, port/service, protocol, and time?*

**Professor requirements covered:** (1) explicit-schema ingestion → typed Parquet; (2) justified cleaning of nulls, duplicates, formats, outliers with row counts; (3) SQL/DataFrame analysis; (4) Docker TP environment.

**Important:** The notebook starts in `DEMO_MODE=True`, expecting **10 invented test flows**, NOT the real dataset. Demo files must be supplied separately. Real inputs are in `data/CSECICIDS2018_improved/` and `data/Service and port name/`; the optional download step below populates those folders. Downloading does not start Spark processing. Results from the demo are not research findings.

## Optional — Download the source datasets

Leave `DOWNLOAD_DATA = False` to use files already on disk. Set it to `True` and run the next cell to download missing datasets and extract the traffic ZIP automatically. Existing source CSVs are preserved. This step uses ordinary Python in your current Jupyter kernel and does not require Docker or PySpark.

On a fresh download, allow at least 50 GiB of free space. Messages indicate downloading and extracting. Rerun the cell after an interruption; incomplete downloads restart from the beginning, while completed CSVs are preserved. Inputs are saved under the project's `data/` folder. Stop after this cell if you only want to obtain the datasets.

In [ ]:
DOWNLOAD_DATA = False  # Optional: change to True to download missing source files.

from pathlib import Path

# Locate the project from notebooks/, the project root, or the outer workspace.
current_folder = Path.cwd().resolve()
root_candidates = [current_folder, *current_folder.parents]
ROOT = next((project for folder in root_candidates
             for project in (folder, folder / "DPA-Volume-Part1-Project")
             if (project / "downloaders" / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder or its notebooks/ folder.")
print("Project folder:", ROOT)

if DOWNLOAD_DATA:
    import runpy
    # Load reusable functions without invoking command-line argument parsing.
    downloader = runpy.run_path(str(ROOT / "downloaders" / "download_data.py"))
    downloader["prepare_iana"](ROOT / "data")
    downloader["prepare_traffic"](ROOT / "data")
    print("Source datasets are ready. No Spark processing was performed.")
else:
    print("Download skipped. Set DOWNLOAD_DATA = True and rerun this cell if needed.")

## 0 — Imports and Spark session
From TP: `SparkSession.builder`, DataFrames and `printSchema()`.

The configured Spark/Java/Python environment runs inside Docker; open its JupyterLab at `http://127.0.0.1:8888/lab`. For Jupyter directly on Windows, the following Spark cells require a compatible local PySpark and Java installation. The optional download cell above does not. Test processing on a small sample before running the full dataset.

In [ ]:
from pathlib import Path
from functools import reduce
import re
import csv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)

spark = (SparkSession.builder
         .master("local[2]")
         .appName("DPA-Volume-SparkSQL")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.ansi.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Spark version:", spark.version)

## 1 — Input paths and demo/full-data toggle
Real inputs use the same folders as the optional downloader: `data/CSECICIDS2018_improved/` and `data/Service and port name/`. The existing IANA filename ending in `(3).csv` is also accepted. Demo inputs are separate and must be supplied under `data/demo/`. Keep `DEMO_MODE=True` until a small-data run is ready; switching to real mode selects all traffic CSVs.

In [ ]:
# ROOT was located in the optional download cell; run that cell even when downloading is disabled.

DEMO_MODE = True  # Real mode selects all CSVs; validate a small-data run first.
if DEMO_MODE:
    flows_dir = ROOT / "data" / "demo" / "cse-cic-ids2018"
    iana_file = ROOT / "data" / "demo" / "iana" / "service-names-port-numbers.csv"
else:
    flows_dir = ROOT / "data" / "CSECICIDS2018_improved"
    iana_folder = ROOT / "data" / "Service and port name"
    iana_file = iana_folder / "service-names-port-numbers.csv"
    if not iana_file.is_file():
        iana_file = next(iter(sorted(iana_folder.glob("service-names-port-numbers*.csv"))), iana_file)
output_root = ROOT / "data" / "parquet" / ("demo" if DEMO_MODE else "real")
report_dir = ROOT / "reports" / ("demo" if DEMO_MODE else "real")
report_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)

csv_paths = sorted(flows_dir.rglob("*.csv"))
assert csv_paths, f"No flow CSV files found in {flows_dir}. Extract the ZIP first."
assert iana_file.exists(), f"Missing IANA CSV: {iana_file}"
print(f"Mode: {'DEMO' if DEMO_MODE else 'REAL'} | Flow files: {len(csv_paths)}")
for p in csv_paths[:12]:
    print(p.relative_to(ROOT))
print("IANA:", iana_file.relative_to(ROOT))

## 2 — Inspect CSV headers, then define an explicit typed schema

The real improved dataset contains many CICFlowMeter fields; headers can vary between releases and files. Unlike the TP's `inferSchema=True`, this notebook explicitly specifies Spark data types. For known fields, we use `IntegerType`, `LongType`, or `DoubleType`; identifiers, labels and timestamp **source strings** are `StringType`. Timestamp conversion happens later with validation. Unrecognized headers receive an **explicit StringType** and are printed for review; check those before reporting results.

**Why inspect only the header?** Reading the first line of each CSV does not load 30 GB into Python memory. Spark reads the file contents.

In [ ]:
def normalized(name):
    return re.sub(r"[^a-z0-9]+", "_", name.strip().lower()).strip("_")

# Explicit type lists for typical CICFlowMeter v3 and improved fields.
TEXT_FIELDS = """flow_id src_ip dst_ip timestamp label attack attempted_category""".split()
INT_FIELDS = """src_port dst_port protocol fwd_psh_flags bwd_psh_flags
 fwd_urg_flags bwd_urg_flags fwd_rst_flags bwd_rst_flags icmp_code icmp_type
 fin_flag_cnt syn_flag_cnt rst_flag_cnt psh_flag_cnt ack_flag_cnt urg_flag_cnt
 cwe_flag_count ece_flag_cnt down_up_ratio""".split()
LONG_FIELDS = """flow_duration tot_fwd_pkts tot_bwd_pkts totlen_fwd_pkts
 totlen_bwd_pkts total_length_of_fwd_packets total_length_of_bwd_packets
 fwd_header_len bwd_header_len fwd_header_length bwd_header_length
 subflow_fwd_pkts subflow_fwd_byts subflow_bwd_pkts subflow_bwd_byts
 fwd_act_data_pkts init_fwd_win_byts init_bwd_win_byts""".split()
DOUBLE_FIELDS = """fwd_pkt_len_max fwd_pkt_len_min fwd_pkt_len_mean fwd_pkt_len_std
 bwd_pkt_len_max bwd_pkt_len_min bwd_pkt_len_mean bwd_pkt_len_std
 flow_byts_s flow_pkts_s flow_iat_mean flow_iat_std flow_iat_max flow_iat_min
 fwd_iat_tot fwd_iat_mean fwd_iat_std fwd_iat_max fwd_iat_min
 bwd_iat_tot bwd_iat_mean bwd_iat_std bwd_iat_max bwd_iat_min
 fwd_pkts_s bwd_pkts_s pkt_len_min pkt_len_max pkt_len_mean pkt_len_std pkt_len_var
 pkt_size_avg fwd_seg_size_avg bwd_seg_size_avg fwd_byts_b_avg fwd_pkts_b_avg
 fwd_blk_rate_avg bwd_byts_b_avg bwd_pkts_b_avg bwd_blk_rate_avg
 fwd_seg_size_min active_mean active_std active_max active_min idle_mean
 idle_std idle_max idle_min""".split()
TYPE_MAP = {**{s: StringType() for s in TEXT_FIELDS},
            **{s: IntegerType() for s in INT_FIELDS},
            **{s: LongType() for s in LONG_FIELDS},
            **{s: DoubleType() for s in DOUBLE_FIELDS}}

def csv_header(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return next(csv.reader(f))

def schema_for(path):
    names = csv_header(path)
    keys = [normalized(n) for n in names]
    if len(keys) != len(set(keys)):
        raise ValueError(f"Duplicate normalized header fields in {path.name}: {names}")
    unknown = [n for n in names if normalized(n) not in TYPE_MAP]
    if unknown:
        print(f"REVIEW explicit types in {path.name}: {unknown}")
    return StructType([StructField(n, TYPE_MAP.get(normalized(n), StringType()), True)
                       for n in names])

for fp in csv_paths:
    sch = schema_for(fp)
    print(fp.name, "|", len(sch), "columns", "|", [(f.name, f.dataType.simpleString()) for f in sch[:7]])

## 3 — Ingestion: CSV with explicit schema → typed Parquet (Bronze)

From TP: reading file sources, `printSchema()`, `.write.mode("overwrite").parquet(...)`. Each CSV uses its own checked header schema; we combine CSVs with `unionByName(allowMissingColumns=True)` in case some days have extra fields.

**Note:** Invalid numeric text may turn into `null` during typed ingestion. In Step 4 we count these missing required values; do **not** silently impute cybersecurity labels or ports.

In [ ]:
def read_typed_file(path):
    return (spark.read
            .schema(schema_for(path))
            .option("header", "true")
            .option("enforceSchema", "false")
            .option("mode", "PERMISSIVE")
            .csv(str(path)))

def canonical_columns(df):
    return df.toDF(*[normalized(c) for c in df.columns])

parts = [canonical_columns(read_typed_file(p)) for p in csv_paths]
flows_ingested = reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), parts)
flows_ingested.printSchema()
flows_ingested.show(5, truncate=False)
bronze_flows_path = str(output_root / "bronze_flows")
flows_ingested.write.mode("overwrite").option("compression", "snappy").parquet(bronze_flows_path)
print("Bronze flows rows:", spark.read.parquet(bronze_flows_path).count())

### IANA secondary data: explicit schema

Port Number stays a string **at ingestion** because IANA includes ranges or blank entries; later we retain only individual integer ports. IANA contains multiple service names per port/protocol: the join must avoid multiplying the number of network flows.

In [ ]:
iana_schema = StructType([
    StructField("Service Name", StringType(), True),
    StructField("Port Number", StringType(), True),
    StructField("Transport Protocol", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Assignee", StringType(), True),
    StructField("Contact", StringType(), True),
    StructField("Registration Date", StringType(), True),
    StructField("Modification Date", StringType(), True),
    StructField("Reference", StringType(), True),
    StructField("Service Code", StringType(), True),
    StructField("Unauthorized Use Reported", StringType(), True),
    StructField("Assignment Notes", StringType(), True),
])
iana_raw = (spark.read.schema(iana_schema).option("header", "true")
            .option("enforceSchema", "false")
            .option("multiLine", "true").csv(str(iana_file)))
iana_raw.printSchema()
bronze_iana_path = str(output_root / "bronze_iana")
iana_raw.write.mode("overwrite").parquet(bronze_iana_path)
print("IANA rows:", spark.read.parquet(bronze_iana_path).count())

## 4 — Cleaning with a step-by-step audit

**Drop vs. impute policy:** We drop records missing destination port, protocol, or label: inventing these would corrupt security conclusions. We also require a parseable timestamp if present for time analysis. We remove exact duplicate rows, impossible port numbers, negative durations, and negative packet counts. We normalize whitespace/case in labels and record modified rows.

**Outlier policy:** Legitimate attacks often have unusually high traffic. We **flag** extreme values but do not remove them merely because they are large. The audit table below records the counts before/after *each cleaning operation*. This is demonstrative until the real CSVs are processed.

In [ ]:
df = spark.read.parquet(bronze_flows_path)
mandatory = {"dst_port", "protocol", "label"}
missing_columns = mandatory - set(df.columns)
if missing_columns:
    raise ValueError(f"Required columns missing: {sorted(missing_columns)}. Check the actual CSV header.")

audit = []
def audit_step(stage, before, after, reason, modified=0):
    audit.append({"step": stage, "rows_before": before, "rows_after": after,
                  "rows_removed": before - after, "rows_modified": modified,
                  "reason": reason})

# TP: withColumn, trim, upper, filtering and datetime conversion
before = df.count()
label_changes = df.filter(
    F.col("label").isNotNull() &
    (F.col("label") != F.upper(F.trim(F.col("label"))))
).count()
df = df.withColumn("label", F.upper(F.trim(F.col("label"))))
audit_step("Normalize label formatting", before, before,
           "Trim spaces and unify capitalization; preserve Attempted labels", label_changes)

if "timestamp" in df.columns:
    event_time = F.coalesce(
        F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("timestamp", "dd/MM/yyyy HH:mm:ss"),
        F.to_timestamp("timestamp", "MM/dd/yyyy HH:mm:ss"),
        F.to_timestamp("timestamp", "dd/MM/yyyy hh:mm:ss a"),
    )
    df = df.withColumn("event_time", event_time)
    failed_time = df.filter(F.col("timestamp").isNotNull() & F.col("event_time").isNull()).count()
    if before and failed_time / before > 0.10:
        raise ValueError(
            f"{failed_time} timestamps failed to parse (>10%). Inspect timestamp formats "
            "before cleaning; do not discard these records blindly."
        )
    audit_step("Parse timestamp", before, before,
               "Convert string timestamp into a typed UTC timestamp", before - failed_time)
    print("Example timestamp strings:")
    df.select("timestamp", "event_time").show(5, truncate=False)

before = df.count()
required_not_null = mandatory | ({"event_time"} if "event_time" in df.columns else set())
cond = F.lit(True)
for name in sorted(required_not_null):
    cond = cond & F.col(name).isNotNull()
    if name == "label":
        cond = cond & (F.length(F.col(name)) > 0)
df = df.filter(cond)
after = df.count()
audit_step("Remove missing critical fields", before, after,
           "Do not impute label, destination port, protocol or required event timestamp")

before = df.count()
df = df.dropDuplicates()
after = df.count()
audit_step("Deduplicate", before, after,
           "Remove exact repeated network flow records")

before = df.count()
valid = (F.col("dst_port").between(0, 65535) &
         F.col("protocol").between(0, 255))
for optional_col in ("src_port",):
    if optional_col in df.columns:
        valid = valid & (F.col(optional_col).isNull() | F.col(optional_col).between(0, 65535))
for optional_col in ("flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts", "totlen_bwd_pkts"):
    if optional_col in df.columns:
        valid = valid & (F.col(optional_col).isNull() | (F.col(optional_col) >= 0))
df = df.filter(valid)
after = df.count()
audit_step("Remove impossible numeric values", before, after,
           "Reject invalid ports/protocols and negative durations/packet counts")

if "event_time" in df.columns:
    before = df.count()
    df = df.filter(F.col("event_time").between("2018-02-01", "2018-03-31 23:59:59"))
    after = df.count()
    audit_step("Expected capture date range", before, after,
               "Keep only flows dated February–March 2018; review any discarded dates")

print("After cleaning:", df.count(), "flows")
df.show(5, truncate=False)

### Save the cleaning audit and the cleaned Parquet
From TP: write/read Parquet and `count()`; for the small **audit report only**, use Python's CSV writer. It is safe to use Python for metadata but not to load the 30 GB traffic CSV into pandas.

In [ ]:
with (report_dir / "cleaning_audit.csv").open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["step", "rows_before", "rows_after", "rows_removed", "rows_modified", "reason"])
    writer.writeheader()
    writer.writerows(audit)

for record in audit:
    print(f"{record['step']}: -{record['rows_removed']} rows | "
          f"modified {record['rows_modified']} | {record['reason']}")

cleaned_path = str(output_root / "silver_cleaned_flows")
df.write.mode("overwrite").option("compression", "snappy").parquet(cleaned_path)
flows_clean = spark.read.parquet(cleaned_path)
print("Verified cleaned Parquet rows:", flows_clean.count())

## 5 — Clean and join IANA registry
From TP: joins and DataFrame aggregations. Keep only single-number ports, normalize `tcp`/`udp` protocol names, and group registry records by `(port, protocol)` so the flow count cannot be multiplied by the join.

**Caution:** An IANA registration does **not** prove that the named application actually ran on that port.

In [ ]:
registry = canonical_columns(spark.read.parquet(bronze_iana_path))
reg_before = registry.count()
reg = (registry
       .select(F.trim(F.col("port_number")).alias("port_text"),
               F.lower(F.trim(F.col("transport_protocol"))).alias("proto_name"),
               F.trim(F.col("service_name")).alias("service_name"))
       .filter(F.col("port_text").rlike("^[0-9]+$"))
       .withColumn("port", F.col("port_text").cast("int"))
       .filter(F.col("port").between(0, 65535))
       .filter(F.col("proto_name").isin("tcp", "udp", "sctp", "dccp"))
       .filter(F.col("service_name").isNotNull() & (F.length("service_name") > 0)))
reg_valid_count = reg.count()
# Some (port, protocol) pairs map to multiple registry entries; choose one alphabetically
# and retain how many service names were associated with that key.
reg_unique = (reg.groupBy("port", "proto_name")
              .agg(F.min("service_name").alias("registered_service"),
                   F.countDistinct("service_name").alias("registered_service_count")))
print("IANA rows raw:", reg_before, "| usable:", reg_valid_count,
      "| unique join keys:", reg_unique.count())

flows_for_join = flows_clean.withColumn(
    "transport_protocol_name",
    F.when(F.col("protocol") == 6, "tcp")
     .when(F.col("protocol") == 17, "udp")
     .when(F.col("protocol") == 132, "sctp")
     .when(F.col("protocol") == 33, "dccp")
     .otherwise(F.lit(None).cast("string"))
)
reg_unique = reg_unique.withColumnRenamed("proto_name", "iana_proto")
flows_enriched = (flows_for_join.alias("flow")
    .join(F.broadcast(reg_unique).alias("iana"),
          (F.col("flow.dst_port") == F.col("iana.port")) &
          (F.col("flow.transport_protocol_name") == F.col("iana.iana_proto")),
          how="left")
    .drop("port", "iana_proto"))

before_join = flows_clean.count()
after_join = flows_enriched.count()
assert before_join == after_join, "IANA join unexpectedly changed the number of network flows"
print("Join preserved rows:", after_join)
flows_enriched.select("dst_port", "transport_protocol_name", "registered_service", "label").show(10, truncate=False)

final_path = str(output_root / "gold_enriched_flows")
flows_enriched.write.mode("overwrite").parquet(final_path)
flows = spark.read.parquet(final_path)

## 6 — Answer the Volume sub-question using DataFrame API + Spark SQL
From TP: `groupBy`, `agg`, `count`, `sum`, `avg`, `sort`, temporary views and SQL. Attempts are **kept as a separate category** rather than silently reclassified as either successful attacks or benign.

In [ ]:
flows = flows.withColumn(
    "traffic_class",
    F.when(F.col("label") == "BENIGN", "Benign")
     .when(F.col("label").contains("ATTEMPTED"), "Attempted")
     .otherwise("Attack")
)
flows.createOrReplaceTempView("network_flows")
print("Traffic classes (DataFrame API):")
flows.groupBy("traffic_class").count().orderBy(F.desc("count")).show()
print("Attack labels (Spark SQL):")
spark.sql("""
SELECT label, COUNT(*) AS flow_count
FROM network_flows
GROUP BY label
ORDER BY flow_count DESC
LIMIT 20
""").show(truncate=False)

In [ ]:
print("Ports and registered services (SQL):")
spark.sql("""
SELECT dst_port,
       COALESCE(registered_service, 'Unmapped / unknown') AS registered_service,
       traffic_class,
       COUNT(*) AS flow_count
FROM network_flows
GROUP BY dst_port, COALESCE(registered_service, 'Unmapped / unknown'), traffic_class
ORDER BY flow_count DESC
LIMIT 20
""").show(truncate=False)

print("Duration/packet comparison (DataFrame API):")
measure_cols = [c for c in ["flow_duration", "tot_fwd_pkts", "tot_bwd_pkts"] if c in flows.columns]
measures = [F.round(F.avg(c), 2).alias(f"avg_{c}") for c in measure_cols]
flows.groupBy("traffic_class").agg(F.count("*").alias("flows"), *measures).show(truncate=False)

print("Time-based patterns:")
if "event_time" in flows.columns:
    (flows.withColumn("day", F.to_date("event_time"))
          .groupBy("day", "traffic_class")
          .count().orderBy("day", "traffic_class")
          .show(30, truncate=False))
else:
    print("No timestamp column found; time analysis skipped.")

if "src_ip" in flows.columns:
    print("Top source IPs by flow count:")
    flows.groupBy("src_ip").count().orderBy(F.desc("count")).show(10)
else:
    print("No src_ip in these CSVs; source-IP analysis not supported by this release.")

## 7 — Track plausible extreme traffic instead of deleting attacks

A large duration or high flow byte rate might be an **attack**, not an error. This is why extreme values are retained and investigated. The threshold here is an illustrative **screening rule**, not a validated intrusion detection model. TP: `filter`, `count`, `explain`.

In [ ]:
if "flow_duration" in flows.columns:
    # Microseconds in CICFlowMeter duration convention: 120,000,000 us = 120 seconds.
    extreme_duration = flows.filter(F.col("flow_duration") > 120_000_000)
    print("Flows with duration greater than 120 seconds (retained):", extreme_duration.count())
else:
    print("No flow_duration column; duration screen skipped")

flows.groupBy("traffic_class").count().explain("formatted")

print("Completed in:", "demo mode" if DEMO_MODE else "real-data mode")
print("Parquet outputs:", output_root)
print("Cleaning audit CSV:", report_dir / "cleaning_audit.csv")

## 8 — Interpretation checklist (complete after using the REAL dataset)

- [ ] Record exact dataset version, number of source files, and actual row counts.
- [ ] Check/adjust any fields printed as `REVIEW explicit types` in Section 2.
- [ ] Inspect actual timestamp samples and date ranges before finalizing cleaning policy.
- [ ] Justify missing-field policy, duplicate removal, and invalid-value filtering with exact audit counts.
- [ ] Report the **difference** between *successful/labelled attacks*, *attempted* attacks, and benign flows.
- [ ] Check that the IANA left join preserved total row count; service names are **registered mappings**, not proven protocol identification.
- [ ] Add a conclusion answering the precise teacher-approved Part 1 sub-question.
- [ ] Include `.explain()` evidence or runtime measurements for selected Spark queries.

**Sources:** [Improved dataset documentation](https://intrusion-detection.distrinet-research.be/CNS2022/CSECICIDS2018.html) and [IANA registry](https://www.iana.org/assignments/service-names-port-numbers/service-names-port-numbers.xhtml).

> Never copy the demo's output counts into the final report. Run it on the downloaded dataset and record measured results.